# Важность признаков для предсказания оценки

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor, Pool
from data_utils import fillna_data

In [8]:
FEATURES = ['program', 'education_level', 'place_type', 'course', 'absence_status', 'discipline', 'module',
       'exam_type', 'grade_10', 'difficulty_avg_score', 'exam_type_prev', 'grade_prev', 'difficulty_prev',
       'avg_grade_prev', 'min_prev', 'max_prev']

CATBOOST_HYPERPARAMETRS = {'iterations': 494, 'learning_rate': 0.3788119430934142, 'loss_function': 'MAE', 'depth': 2, 'l2_leaf_reg': 38.39899891347804, 'min_data_in_leaf': 57}

In [9]:
df = pd.read_csv('total_laggs.csv')
df = fillna_data(df)

X = df[FEATURES]
y1 = df['target_grade']
y2 = df['target_type']

In [10]:
X_train, X_test, y1_train, y1_test = train_test_split(X, y1, test_size=0.3)

In [11]:
cat_features = [col for col in X.columns if X[col].dtype =='str']

train_data = Pool(X_train, y1_train, cat_features)
grade_model = CatBoostRegressor(**CATBOOST_HYPERPARAMETRS , verbose=False)
grade_model.fit(train_data)

CatBoostRegressor(depth=2, iterations=494, l2_leaf_reg=38.39899891347804, learning_rate=0.3788119430934142, loss_function='MAE', min_data_in_leaf=57, verbose=False)

In [12]:
y1_pred = np.round(grade_model.predict(X_test))
test_mae = mean_absolute_error(y1_test, y1_pred)

print('MAE: ', test_mae)

MAE:  1.1587301587301588


In [13]:
columns = X_train.columns
grade_imp = grade_model.feature_importances_

Важность признаков (в процентах):

In [14]:
m = len(columns)

res = {'feature': columns, 'importance for grade': [grade_imp[i] for i in range(m)]}
res = pd.DataFrame(res).sort_values(by=['importance for grade'], ascending=False)

res

,feature,importance for grade
8,grade_10,45.343406
3,course,16.537461
13,avg_grade_prev,5.387008
5,discipline,5.098423
4,absence_status,4.649423
12,difficulty_prev,4.266793
14,min_prev,4.187786
11,grade_prev,4.099650
9,difficulty_avg_score,3.312467
7,exam_type,1.823046


In [11]:
res.to_csv('feature_importance_grade.csv', index = False)